# Bjerknes feedback changes over time

## imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import time

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def window(x):
    return src.utils.get_windowed(x, stride=120)

## Load data

#### $T$, $h$

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True)

### Spatial data

#### Load

In [ ]:
## load spatial data
forced, anom = src.utils.load_consolidated()

## add T,h information
anom = xr.merge([anom, Th])

#### scaling

In [ ]:
hbar_scale = xr.open_dataarray(
    pathlib.Path(SAVE_FP, "cesm_Hbar_scale_v2.nc"),
)

#### max grad

In [ ]:
_, h_mg = src.utils.load_h_data(max_grad=True)

## Compute/plot

In [ ]:
def get_slope_bymonth(x, **kwargs):
    """get slope for each month separately"""
    return x.groupby("time.month").map(src.utils.regress_proj, **kwargs)

### Specify variabes

In [ ]:
## specify plot month
PLOT_MONTH = 1

## specify boxes for fitting
T_VAR = "T_3"
T_FN = src.utils.get_nino3
Tsub_FN = (
    lambda x, H: x.sel(longitude=slice(210, 270))
    .mean("longitude")
    .sel(z_t=H, method="nearest")
)

## specify subsetting funcs
LATS = dict(latitude=slice(-5, 5))
LATS_H = dict(latitude=slice(-5, 5))
LONS_E = dict(longitude=slice(210, 270))
LONS_W = dict(longitude=slice(120, 160))
LONS_TAU = dict(longitude=slice(150, 230))

Set funcs

In [ ]:
## helper func to select and avg
def sel_helper(x, lats, lons):
    """helper func to avg over lats/lons"""

    ## first, average over lons
    x_avg = x.sel(lons).mean("longitude")

    if "latitude" in x_avg.dims:
        x_avg = x_avg.sel(lats).mean("latitude")

    return x_avg


## specify functions
TAU_FN = lambda x: sel_helper(x, LATS, LONS_TAU)
TAU_FN_3 = lambda x: sel_helper(x, LATS, LONS_E)
He_FN = lambda x: sel_helper(x, LATS_H, LONS_E)
Hw_FN = lambda x: sel_helper(x, LATS_H, LONS_W)
Hgrad_FN = lambda x: He_FN(x) - Hw_FN(x)


## specify entrainment / ML averages
LON_AVG = lambda x: x.sel(longitude=slice(210, 270)).mean("longitude")
ENT_AVG = lambda x: x.sel(z_t=slice(50, 80)).mean("z_t")
ML_AVG = lambda x: x.sel(z_t=slice(None, 50)).mean("z_t")

## get T3 volume avg
T3_ENT_AVG = lambda x: ENT_AVG(LON_AVG(x))
T3_ML_AVG = lambda x: ML_AVG(LON_AVG(x))

#### helper funcs

In [ ]:
def regress_over_time(data, x_vars, y_vars):
    """regression over time"""

    ## get windowed data
    data_ = window(data)

    ## empty list to hold coefficients
    coefs = []

    ## shared args
    kwargs = dict(x_vars=x_vars, y_vars=y_vars)

    ## loop thru years
    for year in tqdm.tqdm(data_.year):

        ## get grouped data
        data_y = data_.sel(year=year).groupby("time.month")

        ## do regression
        coefs.append(data_y.map(src.utils.regress_xr_proj, **kwargs))

    return xr.concat(coefs, dim=data_.year)


def regress_wrapper(data, x_vars, y_var, y_fn):
    """regression over time"""

    ## prep data
    y_data = src.utils.reconstruct_wrapper(data[[f"{y_var}", f"{y_var}_comp"]], fn=y_fn)

    ## subset for data
    data_ = xr.merge([data[x_vars], y_data])

    return regress_over_time(data_, x_vars=x_vars, y_vars=[y_var])


def frac_change(x):
    """fractional change"""
    return x / x.isel(year=0) - 1


def make_scatter(anom_, xvar, yvar, month):
    """scatter plot of data for given month"""

    ## prep func
    get_season = lambda x: src.utils.sel_month(
        x.resample({"time": "QS-DEC"}).mean(), month
    )
    prep = lambda x: get_season(x).transpose("time", "member")

    fig, axs = plt.subplots(1, 3, figsize=(8, 2.5), layout="constrained")

    for ax, t_idx in zip(axs, [["1850", "1889"], ["1950", "1989"], ["2061", "2100"]]):

        ## helper func
        prep_ = lambda x: prep(x).sel(time=slice(*t_idx))

        ## get plot data
        plot_data = (prep_(anom_[xvar]), prep_(anom_[yvar]))

        ## get stats
        corr = xr.corr(*plot_data)
        cov = xr.cov(*plot_data)

        ## plot data
        ax.scatter(*plot_data, s=3, label=f"cov = {cov.item():.1e}")

        ax.set_title(f"corr = {corr.item():.2f}")

        ## formatting
        ax_kwargs = dict(ls="--", c="gray", lw=0.5)
        ax.axvline(0, **ax_kwargs)
        ax.axhline(0, **ax_kwargs)
        ax.legend(prop=dict(size=10))

    ## format/scale axes
    src.utils.set_lims(axs)
    axs[1].set_yticks([])
    axs[2].set_yticks([])

    return fig, axs

### Surface terms

#### SST-$\tau_x$

In [ ]:
## compute indices
taux_w = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN)["taux"]
taux_e = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN_3)["taux"]

## regress over time
anom_ = xr.merge([taux_w.rename("taux_w"), taux_e.rename("taux_e"), anom["T_3"]])
mu = regress_over_time(anom_, x_vars=["T_3"], y_vars=["taux_w", "taux_e"]).squeeze(
    drop=True
)

## get coefs
mu_a = mu["taux_w"]
mu_a_prime = mu["taux_e"]

In [ ]:
## specify kwargs
kwargs = dict(anom_=anom_, xvar="T_3", month=12)

for v in ["taux_w", "taux_e"]:
    print(v)
    fig, axs = make_scatter(yvar=v, **kwargs)
    plt.show()

#### $\tau_x$-SSH

In [ ]:
## get tau index
taux_wpac = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN)

## get h indices
get_recon = lambda fn: src.utils.reconstruct_wrapper(anom[["ssh", "ssh_comp"]], fn=fn)
h_e = get_recon(He_FN)["ssh"].rename("h_e")
h_w = get_recon(Hw_FN)["ssh"].rename("h_w")
dh = (h_e - h_w).rename("dh")


## merge data
anom_ = xr.merge([taux_wpac, h_e, h_w, dh])

## regress
beta_h = regress_over_time(anom_, x_vars=["taux"], y_vars=["dh"])
beta_h = beta_h["dh"].squeeze(drop=True)

In [ ]:
## specify kwargs
kwargs = dict(anom_=anom_, xvar="taux", yvar="dh", month=3)
fig, axs = make_scatter(**kwargs)
plt.show()

#### $\tau_x$-$h$

In [ ]:
## get tau index
taux_wpac = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN)

## get gradient of h
Hw_FN_mg = lambda x: sel_helper(x, LATS_H, dict(longitude=slice(120, 180)))
get_dh_mg = lambda x: He_FN(x) - Hw_FN_mg(x)

## get indices
anom_ = xr.merge([taux_wpac, get_dh_mg(h_mg).rename("dh")])


## regress (v1)
beta_h_mg = regress_over_time(data=anom_, x_vars=["taux"], y_vars=["dh"])
beta_h_mg = beta_h_mg["dh"].squeeze(drop=True)

In [ ]:
## specify kwargs
kwargs = dict(anom_=anom_, xvar="taux", yvar="dh", month=9)
fig, axs = make_scatter(**kwargs)
plt.show()

### velocity terms

In [ ]:
## get Hw index
Hw = src.utils.reconstruct_wrapper(anom[["ssh", "ssh_comp"]], fn=Hw_FN)
Hw = Hw.rename({"ssh": "h_w"})

## get tau func
taux_w = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN)["taux"]
taux_e = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN_3)["taux"]

## get U, w in epac
U_epac = src.utils.reconstruct_wrapper(anom[["u", "u_comp"]], fn=T3_ML_AVG)
W_epac = src.utils.reconstruct_wrapper(anom[["w", "w_comp"]], fn=T3_ENT_AVG)

#### $\beta$

In [ ]:
## get data subset
anom_ = xr.merge([Hw, U_epac, W_epac, taux_w.rename("taux_w"), taux_e.rename("taux_e")])

## do regression
beta = regress_over_time(
    data=anom_, x_vars=["taux_w", "taux_e", "h_w"], y_vars=["u", "w"]
)

# ## split by variable
beta_ur = beta["u"].sel(j="taux_w")
beta_ul = beta["u"].sel(j="taux_e")
beta_uh = beta["u"].sel(j="h_w")
beta_wr = beta["w"].sel(j="taux_w")
beta_wl = beta["w"].sel(j="taux_e")
beta_wh = beta["w"].sel(j="h_w")

In [ ]:
## specify kwargs
kwargs = dict(anom_=anom_, yvar="w", month=6)
for xvar in ["taux_w", "taux_e", "h_w"]:
    print(xvar)
    fig, axs = make_scatter(xvar=xvar, **kwargs)
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(beta.month, beta_ur.isel(year=0))
ax.plot(beta.month, beta_ur.isel(year=-1))
plt.show()

#### $\beta_w$

### subsurface terms

#### SSH-$T_{sub}$

In [ ]:
## get ssh index
ssh_epac = src.utils.reconstruct_wrapper(anom[["ssh", "ssh_comp"]], fn=He_FN)
Tsub = src.utils.reconstruct_wrapper(anom[["T", "T_comp"]], fn=T3_ENT_AVG)

## merge data
anom_ = xr.merge([ssh_epac, Tsub])

## regress
a_h = regress_over_time(data=anom_, x_vars=["ssh"], y_vars=["T"])
a_h = a_h["T"].squeeze(drop=True).rename("a_h")

In [ ]:
## specify kwargs
kwargs = dict(anom_=anom_, xvar="ssh", yvar="T", month=12)
fig, axs = make_scatter(**kwargs)
plt.show()

#### $h-T_{sub}$

In [ ]:
## get ssh and T indices
h_epac = He_FN(h_mg)
h_wpac = Hw_FN_mg(h_mg)
Tsub = src.utils.reconstruct_wrapper(anom[["T", "T_comp"]], fn=T3_ENT_AVG)
anom_ = xr.merge([h_epac.rename("h_e"), h_wpac.rename("h_w"), Tsub])

## regress
a_h_mg = regress_over_time(data=anom_, x_vars=["h_e"], y_vars=["T"])
a_h_mg = a_h_mg["T"].squeeze(drop=True).rename("a_h")

In [ ]:
## specify kwargs
kwargs = dict(anom_=anom_, xvar="h_e", yvar="T", month=12)
fig, axs = make_scatter(**kwargs)
plt.show()

#### $\overline{w}$

In [ ]:
## specify mixed layer depth
H0 = 50

## get w_bar
w_bar = src.utils.reconstruct_clim(window(forced[["w", "w_comp"]]))["w"]
is_downwelling = w_bar < 0

## only keep upwelling
w_bar_upwelling = w_bar.where(~(is_downwelling), other=0.0)

## average over Niño 3 and scale by MLD
w_H = 1 / H0 * T3_ENT_AVG(w_bar_upwelling).rename("w_H")

In [ ]:
## plot w to make sure there's no downwelling
fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(w_bar_upwelling.longitude, ENT_AVG(w_bar_upwelling).min(["month", "year"]))
ax.plot(w_bar.longitude, ENT_AVG(w_bar).min(["month", "year"]), ls="--")
ax.axhline(0, lw=0.8, c="gray")
plt.show()

#### $\frac{\partial T}{\partial z}$

In [ ]:
## specify mixed layer depth
H0 = 50

## get Tbar
T_bar = src.utils.reconstruct_clim(window(forced[["T", "T_comp"]]))["T"]

## get dTbar_dz
dTdz_bar = 1 / H0 * (ENT_AVG(T_bar) - ML_AVG(T_bar))
dTdz_bar_v2 = ML_AVG(T_bar.differentiate("z_t"))

## only keep upwelling
is_downwelling = w_bar < 0
dTdz_bar_upwelling = dTdz_bar.where(~(is_downwelling), other=0.0)
dTdz_bar_upwelling_v2 = dTdz_bar_v2.where(~(is_downwelling), other=0.0)

## average over Niño 3
dTdz_bar_T3 = LON_AVG(dTdz_bar_upwelling).rename("dTdz_bar")

In [ ]:
sel = lambda x: x.mean(["year", "month"])

fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(dTdz_bar.longitude, sel(dTdz_bar), label=r"$\Delta_z T$")
ax.plot(dTdz_bar.longitude, sel(dTdz_bar_v2), label=r"$\frac{\partial T}{\partial z}$")

ax_kwargs = dict(c="gray", lw=0.8, ls="--")
ax.axvline(210, **ax_kwargs)
ax.axvline(270, **ax_kwargs)
ax.axhline(0, **ax_kwargs)
ax.legend()
plt.show()

### Consolidate data

#### thermocline feedback

In [ ]:
## should we use ssh or thermocline based h?
use_ssh = True

if use_ssh:
    beta_h_plot = beta_h
    a_h_plot = a_h
    scale = hbar_scale

else:
    beta_h_plot = beta_h_mg
    a_h_plot = a_h_mg
    scale = 1

## apply scaling
beta_h_plot = (beta_h_plot * scale).rename("beta_h")
a_h_plot = (a_h_plot / scale).rename("a_h")

## compute THF
THF = (mu_a * beta_h_plot * a_h_plot * w_H).rename("thf")

## put data in xarray
params = xr.merge([THF, mu_a, beta_h_plot, a_h_plot, w_H])
labels = [r"THF", r"$\mu_a$", r"$\beta_h$", r"$a_h$", r"$w/H$"]
label_dict = dict(zip(list(params), labels))

#### Ekman feedback

In [ ]:
# ## should we use ssh or thermocline based h?
# use_ssh = True

# if use_ssh:
#     beta_h_plot = beta_h
#     a_h_plot = a_h
#     scale = hbar_scale

# else:
#     beta_h_plot = beta_h_mg
#     a_h_plot = a_h_mg
#     scale = 1

# ## apply scaling
# beta_h_plot = (beta_h_plot * scale).rename("beta_h")
# a_h_plot = (a_h_plot / scale).rename("a_h")

# ## compute THF
# THF = (mu_a * beta_h_plot * a_h_plot * w_H).rename("thf")

# ## put data in xarray
# params = xr.merge([THF, mu_a, beta_h_plot, a_h_plot, w_H])
# labels = [r"THF", r"$\mu_a$", r"$\beta_h$", r"$a_h$", r"$w/H$"]
# label_dict = dict(zip(list(params), labels))

### Plots

#### Ekman feedback

##### annual mean

In [ ]:
# ## specify month ranges for plot
# month_ranges = [(1, 12), (2, 5)]


# ## loop through month ranges
# for m_range in month_ranges:

#     print(m_range)

#     ## get difference
#     delta_p = frac_change(params.sel(month=slice(*m_range)).mean("month"))

#     ## make plot
#     fig, axs = plt.subplots(1, 2, figsize=(6, 2.75), layout="constrained")

#     ## plot total
#     axs[0].plot(
#         params.year,
#         delta_p["thf"],
#         c="k",
#         label=label_dict["thf"],
#     )

#     # plot terms
#     for p in list(params)[1:]:
#         axs[1].plot(params.year, delta_p[p], label=label_dict[p])

#     ## format/labeling
#     src.utils.set_lims(axs)
#     axs[1].set_yticks([])
#     ax_kwargs = dict(c="k", lw=0.8, ls="--")
#     for ax in axs:
#         ax.axhline(0, **ax_kwargs)
#         ax.axvline(2020, **ax_kwargs)
#         ax.set_xticks([1870, 2020])
#         ax.legend()

#     plt.show()

#### thermocline feedback

##### annual mean

In [ ]:
## specify month ranges for plot
month_ranges = [(1, 12), (2, 5)]


## loop through month ranges
for m_range in month_ranges:

    print(m_range)

    ## get difference
    delta_p = frac_change(params.sel(month=slice(*m_range)).mean("month"))

    ## make plot
    fig, axs = plt.subplots(1, 2, figsize=(6, 2.75), layout="constrained")

    ## plot total
    axs[0].plot(
        params.year,
        delta_p["thf"],
        c="k",
        label=label_dict["thf"],
    )

    # plot terms
    for p in list(params)[1:]:
        axs[1].plot(params.year, delta_p[p], label=label_dict[p])

    ## format/labeling
    src.utils.set_lims(axs)
    axs[1].set_yticks([])
    ax_kwargs = dict(c="k", lw=0.8, ls="--")
    for ax in axs:
        ax.axhline(0, **ax_kwargs)
        ax.axvline(2020, **ax_kwargs)
        ax.set_xticks([1870, 2020])
        ax.legend()

    plt.show()

##### Seasonal cycle

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(10, 2), layout="constrained")

for ax, p in zip(axs, list(params)):

    ax.set_title(label_dict[p])
    for year in [1870, 2020, 2090]:
        ax.plot(params.month, params[p].sel(year=year))

    ## formatting
    ax.axvspan(1.5, 5.5, alpha=0.1, color="gray", linewidth=0)
    ax.axhline(0, ls="--", c="k", lw=0.8)

plt.show()

In [ ]:
## specify years for comparison
YEARS = (1870, 2030, 2090)

delta_thf1 = params["thf"].sel(year=YEARS[2]) - params["thf"].sel(year=YEARS[1])
delta_thf0 = params["thf"].sel(year=YEARS[1]) - params["thf"].sel(year=YEARS[0])

fig, ax = plt.subplots(figsize=(3, 2.5), layout="constrained")

ax.plot(params.month, delta_thf0, c="k", label=f"{YEARS[1]}-{YEARS[0]}")
ax.plot(params.month, delta_thf1, c="k", label=f"{YEARS[2]}-{YEARS[1]}", ls="--")


## formatting
ax.axvspan(1.7, 5.3, alpha=0.1, color="gray", linewidth=0)
ax.axhline(0, ls="--", c="k", lw=0.8)
ax.legend()
ax.set_title("Diff. b/n periods")
ax.set_xticks([2, 5, 12])

plt.show()